## Step 1 — Verify Java version (must say 11.x.x)

In [ ]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.31.11-hotspot"

In [ ]:
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)
# Must say: openjdk version "11.x.x"
# If it still says 25, go fix JAVA_HOME in System Environment Variables and restart Jupyter

openjdk version "21.0.10" 2026-01-20 LTS
OpenJDK Runtime Environment Microsoft-13106412 (build 21.0.10+7-LTS)
OpenJDK 64-Bit Server VM Microsoft-13106412 (build 21.0.10+7-LTS, mixed mode, sharing)



## Step 2 — Verify PySpark version (must be 3.x.x, not 4.x.x)

In [ ]:
import sys

# Uninstall PySpark 4 and install PySpark 3.5 (stable with Java 11)
# Run this cell ONCE, then comment it out
!{sys.executable} -m pip uninstall pyspark -y
!{sys.executable} -m pip install pyspark==3.5.1 --force-reinstall


# Must say: 3.5.1

Found existing installation: pyspark 3.5.1
Uninstalling pyspark-3.5.1:
  Successfully uninstalled pyspark-3.5.1
  Using cached pyspark-3.5.1-py2.py3-none-any.whl
  Using cached py4j-0.10.9.7-py2.py3-none-any.whl.metadata (1.5 kB)
Using cached py4j-0.10.9.7-py2.py3-none-any.whl (200 kB)
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# Verify versions first
import subprocess, pyspark

java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
print("Java:", java.split("\n")[0])
print("PySpark:", pyspark.__version__)

# Java must say 11.x.x
# PySpark must say 3.5.1

Java: openjdk version "21.0.10" 2026-01-20 LTS
PySpark: 3.5.1


In [ ]:
import os
import sys

# Force correct Java
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Optional sanity check
print("JAVA_HOME =", os.environ["JAVA_HOME"])

# Verify Python executable
print("Python =", sys.executable)

JAVA_HOME = C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot
Python = /usr/local/python/3.12.1/bin/python


In [ ]:
import os

# Use the compatible Java 21 runtime for PySpark in this environment.
os.environ["JAVA_HOME"] = "/usr/local/sdkman/candidates/java/21.0.10-ms"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ":" + os.environ["PATH"]

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

print("Spark started successfully")

# NOTE: Do not stop the Spark session here if you plan to use it in subsequent cells.
# spark.stop()

ConnectionRefusedError: [Errno 111] Connection refused

## Step 3 — Start Spark Session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LiverDiseaseAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark started successfully!")

Spark version: 3.5.1
Spark started successfully!


26/05/07 09:46:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Step 4 — Load the CSV dataset

In [ ]:
import os

# Make sure CSV is in the same folder as this notebook
# Or provide the full path like: r"C:\Users\mooda\Downloads\Training_Liver_Disease_Dataset.csv"
csv_path = r"/workspaces/Big_data_project/Data/5m Sales Records.csv"
# Check if file exists
if not os.path.exists(csv_path):
    print(f"File not found: {csv_path}")
    print("Current directory:", os.getcwd())
    print("Files here:", os.listdir("."))
else:
    print("File found! Loading...")
    df = spark.read.csv(csv_path, header=True, inferSchema=True)
    print("Loaded successfully! Shape:", (df.count(), len(df.columns)))

File found! Loading...


26/05/07 09:46:22 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Loaded successfully! Shape: (5000000, 14)


## Step 5 — Basic Exploration

In [ ]:
# Schema
df.printSchema()

root
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Item Type: string (nullable = true)
 |-- Sales Channel: string (nullable = true)
 |-- Order Priority: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Order ID: integer (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Units Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Unit Cost: double (nullable = true)
 |-- Total Revenue: double (nullable = true)
 |-- Total Cost: double (nullable = true)
 |-- Total Profit: double (nullable = true)



In [ ]:
# First 5 rows
df.show(5, truncate=False)

+----------------------------+-------+---------------+-------------+--------------+----------+---------+---------+----------+----------+---------+-------------+----------+------------+
|Region                      |Country|Item Type      |Sales Channel|Order Priority|Order Date|Order ID |Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|
+----------------------------+-------+---------------+-------------+--------------+----------+---------+---------+----------+----------+---------+-------------+----------+------------+
|Australia and Oceania       |Palau  |Office Supplies|Online       |H             |3/6/2016  |517073523|3/26/2016|2401      |651.21    |524.96   |1563555.21   |1260428.96|303126.25   |
|Europe                      |Poland |Beverages      |Online       |L             |4/18/2010 |380507028|5/26/2010|9340      |47.45     |31.79    |443183.0     |296918.6  |146264.4    |
|North America               |Canada |Cereal         |Online       |M      

In [ ]:
# Basic statistics
df.describe().show()

26/05/07 09:47:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------+----------+-------------+--------------+----------+-------------------+---------+------------------+------------------+------------------+------------------+------------------+------------------+
|summary|            Region|    Country| Item Type|Sales Channel|Order Priority|Order Date|           Order ID|Ship Date|        Units Sold|        Unit Price|         Unit Cost|     Total Revenue|        Total Cost|      Total Profit|
+-------+------------------+-----------+----------+-------------+--------------+----------+-------------------+---------+------------------+------------------+------------------+------------------+------------------+------------------+
|  count|           5000000|    5000000|   5000000|      5000000|       5000000|   5000000|            5000000|  5000000|           5000000|           5000000|           5000000|           5000000|           5000000|           5000000|
|   mean|              NULL|       NULL|      NULL|     

In [ ]:
# Check null values per column
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show()

+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+
|Region|Country|Item Type|Sales Channel|Order Priority|Order Date|Order ID|Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|
+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+
|     0|      0|        0|            0|             0|         0|       0|        0|         0|         0|        0|            0|         0|           0|
+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+



In [ ]:
# Stop Spark when done
# NOTE: Do not stop Spark until all subsequent cells have finished running.
# spark.stop()
print("Spark session is still active. Stop it after all analysis is complete.")

Spark session is still active. Stop it after all analysis is complete.


#Data Preprocessing

In [ ]:
!pip install numpy pandas matplotlib scikit-learn


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    MultilayerPerceptronClassifier
)

from pyspark.ml.clustering import KMeans

print("All imports successful!")

All imports successful!


In [ ]:
region_indexer = StringIndexer(
    inputCol="Region",
    outputCol="RegionIndex"
)
item_indexer = StringIndexer(
    inputCol="Item Type",
    outputCol="ItemTypeIndex"
)
df = region_indexer.fit(df).transform(df)
df = item_indexer.fit(df).transform(df)

df.show(5)

+--------------------+-------+---------------+-------------+--------------+----------+---------+---------+----------+----------+---------+-------------+----------+------------+-----------+-------------+
|              Region|Country|      Item Type|Sales Channel|Order Priority|Order Date| Order ID|Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|RegionIndex|ItemTypeIndex|
+--------------------+-------+---------------+-------------+--------------+----------+---------+---------+----------+----------+---------+-------------+----------+------------+-----------+-------------+
|Australia and Oce...|  Palau|Office Supplies|       Online|             H|  3/6/2016|517073523|3/26/2016|      2401|    651.21|   524.96|   1563555.21|1260428.96|   303126.25|        5.0|          0.0|
|              Europe| Poland|      Beverages|       Online|             L| 4/18/2010|380507028|5/26/2010|      9340|     47.45|    31.79|     443183.0|  296918.6|    146264.4|        1.0|

In [ ]:
df = df.dropna()
print("Rows after cleaning:", df.count())

AssertionError: Undefined error message parameter for error class: CANNOT_PARSE_DATATYPE. Parameters: {'error': '[Errno 111] Connection refused'}